In [ ]:
#https://sdw-2023-prd.up.railway.app/swagger-ui/index.html#/Users%20Controller/findById
sdw2023_api_url = 'https://sdw-2023-prd.up.railway.app'

In [ ]:
#Extração dos Id's
import pandas as pd


df = pd.read_csv('SDW2023.csv')
user_ids = df['UserId'].tolist()
print(user_ids)



In [ ]:
import requests
import json

def get_user(id):
  response = requests.get(f'{sdw2023_api_url}/users/{id}')
  return response.json() if response.status_code == 200 else None

users = [user for id in user_ids if (user := get_user(id)) is not None]
print(json.dumps(user, indent=2))

In [ ]:
import openai

openai.api_key = openai_api_key

def generate_ai_news(user):
  completion = openai.ChatCompletion.create(
      model =  "gpt-4o-mini",
      messages=[
          {
              "role": "system",
              "content": "Você é um especialista em marketing bancário."
          },
          {
              "role": "user",
              "content": f"Crie uma mensagem para {user['name']} sobre a importância dos investimentos (máximo de 100 caracteres)"
          }
      ]
  )

  responseChatGPT = completion.choices[0].message.content.strip('\"')

  return responseChatGPT

for user in users:
    news = generate_ai_news(user)
    print(news)
    user['news'].append({
        "icon": "https://digitalinnovationone.github.io/santander-dev-week-2023-api/icons/credit.svg",
        "description": news
    })

In [ ]:
def update_user(user):
  response = requests.put(f"{sdw2023_api_url}/users/{user['id']}", json=user)
  return True if response.status_code == 200 else False

for user in users:
  success = update_user(user)
  print(f"User {user['name']} updated? {success}!")
  